# Spark Application Logs
A proper application must have some kind of application logging.

## How to use Log4j with pyspark?
Configuring Log4j is a three step process :
- Create a Log4j configuration file.
- Configure Spark JVM to pickup the Log4j configuration file.
- Create a python class to get Spark's Log4j instance and use it in the pyspark program

## Log4j properties file
Log4j works almost the same way as python logs.
### Componenets of Log4j : 
- Logger : It is a set of apis which we are going to use in our application.
- Configurations : This is defined in the Log4j properties file and they are loaded with the loggers at run time.
    - Log4j configurations are defined in the hierarchy and the top most hierarchy is the root category.
    - For any hierarchy or category we define two things first is the log level and the second thing is the list of appenders.
    - Log4j supports multiple log levels such as INFO,DEBUG,WARN,ERROR
- Appender : Appenders are the output destinations such as console and log file. These appenders are also configured in the Log4j property file.
    - Console appender : 
- **NOTE :** Hierarchy and Appender section define the root level Log4j configurations and they will stop all the log messages sent by the spark and other packages except Warning and errors.


#### Automatically detect the configuration file and set up the configuration for Log4j for a spark program written in python
In this pyspark application with dynamic project level log4j logging configured here in this example is portable in nature because the log4j.properties file along with the logs folder stays inside the project's folder which makes it extremely portable

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand

# For keeping track of time taken to complete the spark computation task
import time
import sys

# spark Log4j logging related imports
from pyspark import SparkConf
import os

# Logging related Spark configurations setup
# log4k.properties configuration file path setup
# Determine project directory — works in both script & notebook
try:
    project_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # __file__ is not defined in interactive mode (e.g., Jupyter)
    project_dir = os.getcwd()

log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")

# Create SparkConf with custom Log4j config
conf = (
    SparkConf()
    .setAppName("CPU_Stress_Test")
    .setMaster("local[*]")
     # JVM property for log4j v1
    .set("spark.driver.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")
    .set("spark.executor.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")
    # Ensure executors also get it via --files equivalent
    .set("spark.files", log4j_config_path)
)


start = time.time()

#Create Spark session in local mode using all cores
spark = SparkSession.builder \
    .config(conf=conf)\
    .getOrCreate()

print("Spark master:", spark.sparkContext.master)
print("Total cores Spark sees:", spark.sparkContext.defaultParallelism)

#Create a large synthetic dataset (e.g., 100 million rows)
num_rows = 99_999
num_partitions = spark.sparkContext.defaultParallelism  # same as CPU cores

df = spark.range(0, num_rows, numPartitions=num_partitions) \
           .withColumn("random_val", rand())

# I want to know the amount of Ram taken by the dataFrame in Mbs
# converting df to rdd rows
rdd = df.rdd.map(lambda row: row.asDict())
# Estimate memory size of one partition
def estimate_partition_size(partition):
    import sys
    size = 0
    for record in partition:
        size += sys.getsizeof(record)
    yield size

partition_sizes = rdd.mapPartitions(estimate_partition_size).collect()
total_bytes = sum(partition_sizes)
total_mb = total_bytes / (1024 * 1024)

#Apply heavy transformations — wide operations
#    Force Spark to use multiple stages and shuffles
aggregated_df = (
    df.withColumn("squared", col("random_val") * col("random_val"))
      .groupBy((col("id") % 100).alias("group"))  # 100 groups
      .avg("squared")                            # aggregation
      .orderBy("group")                          # shuffle operation
)

#Trigger computation (action)
aggregated_df.show()

end = time.time()

time_taken_in_sec = end - start
time_taken_in_min = (end - start) / 60
print(f"""
        Spark task complete!
        Time taken in seconds = {time_taken_in_sec}
        Time taken in minutes = {time_taken_in_min}
        Estimated DataFrame size in memory: {total_mb:.2f} MB
""")


Setting default log level to "DEBUG".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/10/29 10:47:39 INFO Server: jetty-11.0.24; built: 2024-08-26T18:11:22.448Z; git: 5dfc59a691b748796f922208956bd1f2794bcd16; jvm 17.0.15+6-Ubuntu-0ubuntu120.04
25/10/29 10:47:39 INFO Server: Started Server@73b4d8b7{STARTING}[11.0.24,sto=30000] @1934ms
25/10/29 10:47:39 INFO AbstractConnector: Started ServerConnector@3f1256c8{HTTP/1.1, (http/1.1)}{0.0.0.0:4040}
25/10/29 10:47:39 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@10d377b4{/,null,AVAILABLE,@Spark}
25/10/29 10:47:39 INFO ContextHandler: Stopped o.s.j.s.ServletContextHandler@10d377b4{/,null,STOPPED,@Spark}
25/10/29 10:47:39 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@16a4e6e3{/jobs,null,AVAILABLE,@Spark}
25/10/29 10:47:39 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@2f0d9e56{/jobs/json,null,AVAILABLE,@Spark}
25/10/29 10:47:39 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@4bc7147{/jobs/job,null,AVAILABLE,@Spark}
25/10/29 10:47:39 INFO ContextHandler: Started o.s.j.s.ServletC

+-----+-------------------+
|group|       avg(squared)|
+-----+-------------------+
|    0| 0.3340439263220004|
|    1| 0.3344921165066457|
|    2|0.34502446565187284|
|    3|  0.342198436629548|
|    4|0.34417533677080037|
|    5|0.33386892671429463|
|    6|  0.347319529342005|
|    7|0.31667741285428913|
|    8| 0.3379351474029461|
|    9|0.34771968288406335|
|   10| 0.3420210826290725|
|   11| 0.3386432896372724|
|   12|0.33138876521252214|
|   13|0.34514370733520816|
|   14|0.32780599370701674|
|   15| 0.3483815351712168|
|   16| 0.3312397757623714|
|   17| 0.3297252764147367|
|   18|0.33761620820494354|
|   19|0.33892415565261996|
+-----+-------------------+
only showing top 20 rows

        Spark task complete!
        Time taken in seconds = 7.9841554164886475
        Time taken in minutes = 0.13306925694147745
        Estimated DataFrame size in memory: 17.55 MB

